# Precision and Devices

PyTorch users have a familiar runtime error pattern:

```python
x = torch.tensor([1.0, 2.0], dtype=torch.float64).to("mps")
# RuntimeError: Cannot convert a MPS Tensor to float64 dtype as the MPS
# framework doesn't support float64. Please use float32 instead.
```

MLX dropped float64 support on Metal GPU in version 0.31, and you only
find out at op-launch time — deep in a forward pass, after training has
already started. Same problem class as the shape errors we saw in
[Tutorial 01](01_tensors_and_types.ipynb): a hardware/library constraint
the user could have known about, surfaced as a stack trace.

idris-ml's `Tensor` now carries both a **device** and a **dtype** at the
type level, and a `Compatible (device, dtype)` capability check makes
the (Metal GPU, float64) combination a compile error. PyTorch's runtime
`RuntimeError` lifted to the type system.


## The four-parameter Tensor

After the `dtype` work landed (2026-05-17), the `Tensor` record carries
four 0-quantity phantom parameters:

```idris
record Tensor (dims : Vect rank Nat) (0 d : Device) (0 dt : DType) (0 g : GradMode) where
  constructor MkTensor
  tensorPtr : AnyPtr
  paramId   : Maybe String
```

- `dims` — the shape, the usual dependently-typed Vect.
- `d` — the device (CPU, CUDA n, MPS, MlxDev MGpu, MlxDev MCpu, BYO).
- `dt` — the dtype (F32, F64, BF16, F16, IntN n, UInt n, Bool).
- `g` — the grad mode (WithGrad vs NoGrad).

All four are erased at runtime (the `0` quantity). They exist purely to
let the compiler prove things about tensors before the program runs.


## The `Compatible` table

Which (device, dtype) pairs are allowed? It's an empty marker interface
— the existence of an instance IS the proof:

```idris
public export
interface Compatible (0 d : Device) (0 t : DType) where
```

The instance table for the built-in devices:

| Device         | F64 | F32 |
|----------------|-----|-----|
| `CPU`          | ✓   |     |
| `TapeDev`      | ✓   |     |
| `TorchDev`     | ✓   |     |
| `MlxDev MCpu`  | ✓   | ✓   |
| `MlxDev MGpu`  |     | ✓   |

The one deliberately missing cell — `(MlxDev MGpu, F64)` — is where the
demo error lives.


In [ ]:
:doc Compatible

### Positive case

A tensor on the MLX GPU stream at F32 compiles cleanly. The compiler
finds `Compatible (MlxDev MGpu) F32` and elaboration succeeds.


In [ ]:
the (IO (Tensor [4] (MlxDev MGpu) F32 WithGrad)) ?someBuilder

### Negative case

Asking for F64 on the same device fails to typecheck. There is no
`Compatible (MlxDev MGpu) F64` instance, so the constraint search at
the call site fails. The error message points at the spelling site —
exactly where the user can fix it.


In [ ]:
the (IO (Tensor [4] (MlxDev MGpu) F64 WithGrad)) ?someBuilder
-- Expected error:
-- Can't find an implementation for Compatible (MlxDev MGpu) F64

## Parametric dtype families

The dtype tags are `Nat`-parameterized type constructors, not opaque
names. `F32` is just an alias:

```idris
data Float : Nat -> Type where MkFloat : Float n
F32 : Type
F32 = Float 32
F64 : Type
F64 = Float 64
F16 : Type
F16 = Float 16
```

And there are four families:

- `Float n` — IEEE 754 floats (F32 = 32, F64 = 64, F16 = 16).
- `BFloat n` — brain-float (BF16 = BFloat 16). Distinct from `Float`
  because the mantissa/exponent layout is incomparable.
- `IntN n` — signed integers (I8, I16, I32, I64). Idris reserves the
  bare name `Int`, hence the `N` suffix on the family constructor.
- `UInt n` — unsigned integers (U8, U16, U32).
- Plus a non-parametric `Bool` for boolean masks.


In [ ]:
:t Float 64
:t F32
:t IntN 32
:t I8
:doc IsDType

## `UpcastableTo` — derived lossless conversion

When can one tensor type be safely converted to another without losing
information? Within a single dtype family, the answer is monotonic on
bit width: every `Float 32` fits in a `Float 64`, every `Int 16` fits
in `Int 32`. Across families, it's complicated — a `UInt 8` value fits
in `F16`'s mantissa bit-wise, but the type system can't know whether
the user meant an ordinal label (where the cast is type confusion) or
a magnitude (where it's lossless).

idris-ml resolves this by **deriving within-family upcasts** and
**forbidding cross-family auto-derivation**:

```idris
{m, n : Nat} -> LTE m n => UpcastableTo (Float m) (Float n) where
{m, n : Nat} -> LTE m n => UpcastableTo (BFloat m) (BFloat n) where
{m, n : Nat} -> LTE m n => UpcastableTo (IntN m) (IntN n) where
{m, n : Nat} -> LTE m n => UpcastableTo (UInt m) (UInt n) where
```

Idris's auto-search synthesises the `LTE m n` proof from the `Nat`
constructors at the call site.


In [ ]:
-- F32 -> F64 is lossless (LTE 32 64 is provable)
the (UpcastableTo F32 F64) %search

-- I8 -> I64 is lossless (LTE 8 64)
the (UpcastableTo I8 I64) %search

-- BF16 -> BFloat 32 is lossless within the brain-float family
the (UpcastableTo BF16 (BFloat 32)) %search

In [ ]:
-- F64 -> F32 is narrowing (LTE 64 32 has no proof)
the (UpcastableTo F64 F32) %search
-- Expected error: Can't find an implementation for LTE 64 32

-- BF16 -> F32 crosses families (no instance for BFloat -> Float)
the (UpcastableTo BF16 F32) %search
-- Expected error: Can't find an implementation for
--   UpcastableTo (BFloat 16) (Float 32)

## `tcast` — explicit narrowing

Sometimes you genuinely want to narrow F64 → F32, or cast across
families. The `tcast` function takes the target dtype as an explicit
argument, making the caller's intent visible at the call site:

```idris
tcast : (0 to : DType) -> (IsDType from, IsDType to) =>
        Tensor dims d from g -> IO (Tensor dims d to g)

tcastSafe : (UpcastableTo from to, IsDType from, IsDType to) =>
            Tensor dims d from g -> IO (Tensor dims d to g)
```

Use `tcastSafe` for the common upcast case — the compiler verifies
via `UpcastableTo` that no information is lost. Use `tcast` only when
the conversion is deliberately narrowing or cross-family; the call
itself is the explicit signal that the caller takes responsibility.

(Runtime support for both is in flight — the type signatures ship
today, the C-side `tensor_cast_dtype` primitive lands in a follow-up.)


## Build-mode targeting

Idris-2 can't drive type-level selection from a runtime env var —
types are fixed at elaboration time, before `main` runs. So how do
examples switch between F64 mode and F32-on-MLX-GPU mode?

Build-time generated source. The Makefile reads `BACKEND` +
`MLX_DEVICE` and emits `BuildConfig.idr`:

```idris
module BuildConfig

public export
ExampleDevice : Type
ExampleDevice = CPU            -- or MlxDev MGpu in F32 mode

public export
ExampleDType : DType
ExampleDType = F64             -- or F32 in F32 mode
```

Examples import `BuildConfig` and reference `ExampleDevice` /
`ExampleDType` instead of hardcoded `CPU` / `F64`:

```idris
model : Network 2 [] 3 ExampleDevice ExampleDType WithGrad
```

Switching modes is a Makefile flip:

```bash
make BACKEND=tape install              # F64 mode (default)
make BACKEND=mlx MLX_DEVICE=gpu install  # F32 mode on Metal GPU
```

No example source edits required. The library stays fully polymorphic
in `dt`; the callers (examples) pin the concrete dtype at the leaf.


## Summary

| | PyTorch | idris-ml |
|---|---------|----------|
| Dtype tracking | Runtime (`.dtype` property) | Compile-time (phantom type) |
| Device-dtype mismatch | `RuntimeError` at op launch | Type error before compilation |
| Lossless upcast | Implicit, sometimes silent | Explicit via `UpcastableTo` |
| Narrowing | Implicit, no warning | Requires explicit `tcast` |
| Cross-family cast | Implicit, sometimes silent | Requires explicit `tcast` |
| Multi-mode targeting | Code change | Build flag |

The cost at runtime is zero — the dtype slot is `0`-quantity, erased
before code generation. The only thing the user pays is one more
type parameter on `Tensor` and one constraint at construction sites.

Next: back to [01 Tensors and Types](01_tensors_and_types.ipynb) for
the basics, or [06 Device Safety](06_device_safety.ipynb) for the
device-only story this notebook builds on.
